In [1]:
import torch
from transformers import RobertaTokenizer,RobertaModel,RobertaForMaskedLM, pipeline, BertForMaskedLM, BertTokenizer, BertModel, AlbertConfig, AlbertModel, AlbertTokenizer, AlbertForMaskedLM , TFAutoModel, AutoModelForMaskedLM, AutoTokenizer, AutoModelForCausalLM
import math
import numpy as np
import json
import random
import pandas as pd
import tensorflow as tf
from torch.nn import CrossEntropyLoss
from tqdm.notebook import tqdm
import re
from scipy.optimize import linear_sum_assignment

In [2]:
import logging
import os
import sys
logging.basicConfig(level=logging.INFO)
import warnings
warnings.filterwarnings('ignore')

In [3]:
#Data 
def prep_data(source, words):
    source = "test/"+source+".txt"
    file = open(source, "r", encoding = 'utf-8')
    lines = file.readlines()

    data = []
    data_3line = []

    for l in range(12):
        line = lines[l]

        data.append(line)

        prev = lines[l-1].replace("{}", words[l-1]) if l > 0 else ""
        next = lines[l+1].replace("{}", words[l+1]) if l < 11 else ""

        data_3line.append(prev + line + next)
        
    return [data, data_3line]

In [4]:
def uni_predict(text, model, tokenizer):
    # Tokenized input
    # text = "[CLS] I got restricted because Tom reported my reply [SEP]"
    text = text
    tokenized_text = tokenizer.tokenize(text)
    sentence_score = 0
    indexed_tokens = tokenizer.convert_tokens_to_ids(tokenized_text)
    length = len(tokenized_text)
    tokens_tensor = torch.tensor([indexed_tokens])
    tokens_tensor = tokens_tensor.to('cuda')
    #masked_tensor = torch.tensor([masked_index])
    with torch.no_grad():
        outputs = model(tokens_tensor, labels= tokens_tensor)
    loss = outputs[0]
    sentence_score = -loss
    return sentence_score

In [5]:
def greedy_select(df):
    selected_positions = []
    remaining_rows = set(df.index)
    remaining_columns = set(df.columns)

    while len(remaining_rows) > 0 and len(remaining_columns) > 0:
        min_value = float('inf')
        min_position = None

        # Find the smallest value and its position
        for row in remaining_rows:
            for column in remaining_columns:
                value = df.at[row, column]
                if value < min_value:
                    min_value = value
                    min_position = (row, column)

        # Remove the row and column
        remaining_rows.remove(min_position[0])
        remaining_columns.remove(min_position[1])

        # Add the position to the selected list
        selected_positions.append(min_position)

    return sorted(selected_positions, key=lambda x: x[0])

In [6]:
def score_model(model, tokenizer, data, opts):
    df = pd.DataFrame()
    t1_score = 0
    t3_score = 0
    for d in tqdm(data):
        correct = opts[data.index(d)]
        #print(d)
        #print("Correct option is: ", correct)
        scores = {}
        for o in opts:
            sentence = d.replace("{}", o)
            scores.update({o : float(uni_predict(sentence, model, tokenizer).item())})
        df = df.append(scores, ignore_index=True)
        scores = sorted(scores.items(), key=lambda x: x[1], reverse = True)
        i = 0
        for key, value in scores:
            #print(key, ':', value)
            t1_score += ((i == 0) and (key == correct))
            t3_score += ((i < 3) and (key == correct))
            i += 1
        #print()
    
    print("Top 1 Score:", t1_score/12)
    print("Top 3 Score:", t3_score/12)
    
    #print(df)
    df = df.apply(lambda row: row / row.mean(), axis=1)
    x,y = linear_sum_assignment(df)
    out = pd.DataFrame({'Word': df.columns[y], 'Sentence': df.index[x]})
    #print(out)
    final_score = 0
    for n in range(12):
        final_score += (opts[n] == out.iloc[n]['Word'])
    final_score = final_score/12
    print("Forced Choiced Combinatorially Optimized Score:",final_score)
    greedy_values = greedy_select(df) 
    #print("Greedy Min Values:",greedy_values)
    final_score = 0
    for n in range(12):
        final_score += (opts[n] == greedy_values[n][1])
    final_score = final_score/12
    print("Forced Choiced Greedy Algorithm Score:",final_score)

In [12]:
def run_tests(model, tokenizer):
    words = {
        "Polish" : ["stały","dzieci","dalszy","trudno","bawi","krajach","pieniądze","kosztowne","młode","wierzą","osiąga","trenowania"],
        "Slovak" : ["vyrástli","deti","druhoradé","ťažké","hrá","krajinách","peniaze","drahé","nevyvinuté","veria","dostane","trénovania"],
        "Czech"  : ["vyrostli","děti","druhořadé","těžké","hraje","zemích","peníze","drahé","nevyvinuté","věří","dostane","trénování"]
    }
    langs = ["Polish","Czech","Slovak"]
    for lang in tqdm(langs):
        print(lang)
        data = prep_data(lang, words[lang])
        print("1 Line Data:")
        score_model(model, tokenizer, data[0], words[lang])
        print("3 Line Data:")
        score_model(model, tokenizer, data[1], words[lang])
        print()

In [13]:
#Polish
model = BertForMaskedLM.from_pretrained("dkleczek/bert-base-polish-uncased-v1",ignore_mismatched_sizes=True).cuda()
tokenizer = BertTokenizer.from_pretrained("dkleczek/bert-base-polish-uncased-v1")
run_tests(model, tokenizer)

Some weights of the model checkpoint at dkleczek/bert-base-polish-uncased-v1 were not used when initializing BertForMaskedLM: ['cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


  0%|          | 0/3 [00:00<?, ?it/s]

Polish
1 Line Data:


  0%|          | 0/12 [00:00<?, ?it/s]

Top 1 Score: 0.4166666666666667
Top 3 Score: 0.75
Forced Choiced Combinatorially Optimized Score: 0.5
Forced Choiced Greedy Algorithm Score: 0.4166666666666667
3 Line Data:


...

Top 1 Score: 0.08333333333333333
Top 3 Score: 0.16666666666666666
Forced Choiced Combinatorially Optimized Score: 0.16666666666666666
Forced Choiced Greedy Algorithm Score: 0.08333333333333333

Slovak
1 Line Data:


  0%|          | 0/12 [00:00<?, ?it/s]

Top 1 Score: 0.08333333333333333
Top 3 Score: 0.5
Forced Choiced Combinatorially Optimized Score: 0.0
Forced Choiced Greedy Algorithm Score: 0.16666666666666666
3 Line Data:


  0%|          | 0/12 [00:00<?, ?it/s]

Top 1 Score: 0.0
Top 3 Score: 0.0
Forced Choiced Combinatorially Optimized Score: 0.0
Forced Choiced Greedy Algorithm Score: 0.0



In [14]:
#Czech 2
#Uses BERT tokenizer to avoid sentencepiece "not a string" error
tokenizer = BertTokenizer.from_pretrained("UWB-AIR/Czert-A-base-uncased", from_tf=True)
model = AlbertForMaskedLM.from_pretrained("UWB-AIR/Czert-A-base-uncased", from_tf=True).cuda()
run_tests(model, tokenizer)

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'AlbertTokenizer'. 
The class this function is called from is 'BertTokenizer'.
All TF 2.0 model weights were used when initializing AlbertForMaskedLM.

Some weights of AlbertForMaskedLM were not initialized from the TF 2.0 model and are newly initialized: ['predictions.decoder.weight', 'predictions.decoder.bias', 'predictions.decoder.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  0%|          | 0/3 [00:00<?, ?it/s]

Polish
1 Line Data:


  0%|          | 0/12 [00:00<?, ?it/s]

Top 1 Score: 0.08333333333333333
Top 3 Score: 0.25
Forced Choiced Combinatorially Optimized Score: 0.0
Forced Choiced Greedy Algorithm Score: 0.08333333333333333
3 Line Data:


...

Top 1 Score: 0.3333333333333333
Top 3 Score: 0.8333333333333334
Forced Choiced Combinatorially Optimized Score: 0.5833333333333334
Forced Choiced Greedy Algorithm Score: 0.4166666666666667

Slovak
1 Line Data:


  0%|          | 0/12 [00:00<?, ?it/s]

Top 1 Score: 0.0
Top 3 Score: 0.16666666666666666
Forced Choiced Combinatorially Optimized Score: 0.08333333333333333
Forced Choiced Greedy Algorithm Score: 0.0
3 Line Data:


  0%|          | 0/12 [00:00<?, ?it/s]

Top 1 Score: 0.25
Top 3 Score: 0.25
Forced Choiced Combinatorially Optimized Score: 0.0
Forced Choiced Greedy Algorithm Score: 0.08333333333333333



In [15]:
tokenizer = RobertaTokenizer.from_pretrained('gerulata/slovakbert')
model = RobertaForMaskedLM.from_pretrained('gerulata/slovakbert').cuda()
run_tests(model, tokenizer)

  0%|          | 0/3 [00:00<?, ?it/s]

Polish
1 Line Data:


  0%|          | 0/12 [00:00<?, ?it/s]

Top 1 Score: 0.08333333333333333
Top 3 Score: 0.3333333333333333
Forced Choiced Combinatorially Optimized Score: 0.08333333333333333
Forced Choiced Greedy Algorithm Score: 0.0
3 Line Data:


  0%|          | 0/12 [00:00<?, ?it/s]

...

Top 1 Score: 0.0
Top 3 Score: 0.4166666666666667
Forced Choiced Combinatorially Optimized Score: 0.5
Forced Choiced Greedy Algorithm Score: 0.4166666666666667

Slovak
1 Line Data:


  0%|          | 0/12 [00:00<?, ?it/s]

Top 1 Score: 0.25
Top 3 Score: 0.3333333333333333
Forced Choiced Combinatorially Optimized Score: 0.25
Forced Choiced Greedy Algorithm Score: 0.25
3 Line Data:


  0%|          | 0/12 [00:00<?, ?it/s]

Top 1 Score: 0.08333333333333333
Top 3 Score: 0.5833333333333334
Forced Choiced Combinatorially Optimized Score: 0.6666666666666666
Forced Choiced Greedy Algorithm Score: 0.25



In [16]:
tokenizer = AutoTokenizer.from_pretrained("Milos/slovak-gpt-j-1.4B")
model = AutoModelForCausalLM.from_pretrained("Milos/slovak-gpt-j-1.4B").cuda()
run_tests(model, tokenizer)

  0%|          | 0/3 [00:00<?, ?it/s]

Polish
1 Line Data:


  0%|          | 0/12 [00:00<?, ?it/s]

Top 1 Score: 0.4166666666666667
Top 3 Score: 0.6666666666666666
Forced Choiced Combinatorially Optimized Score: 0.5833333333333334
Forced Choiced Greedy Algorithm Score: 0.4166666666666667
3 Line Data:


  0%|          | 0/12 [00:00<?, ?it/s]

...

Top 1 Score: 0.9166666666666666
Top 3 Score: 1.0
Forced Choiced Combinatorially Optimized Score: 1.0
Forced Choiced Greedy Algorithm Score: 1.0

Slovak
1 Line Data:


  0%|          | 0/12 [00:00<?, ?it/s]

Top 1 Score: 1.0
Top 3 Score: 1.0
Forced Choiced Combinatorially Optimized Score: 1.0
Forced Choiced Greedy Algorithm Score: 1.0
3 Line Data:


  0%|          | 0/12 [00:00<?, ?it/s]

Top 1 Score: 1.0
Top 3 Score: 1.0
Forced Choiced Combinatorially Optimized Score: 1.0
Forced Choiced Greedy Algorithm Score: 1.0



In [17]:
tokenizer = AutoTokenizer.from_pretrained("sdadas/polish-gpt2-small")
model = AutoModelForCausalLM.from_pretrained("sdadas/polish-gpt2-small").cuda()
run_tests(model, tokenizer)

  0%|          | 0/3 [00:00<?, ?it/s]

Polish
1 Line Data:


  0%|          | 0/12 [00:00<?, ?it/s]

Top 1 Score: 1.0
Top 3 Score: 1.0
Forced Choiced Combinatorially Optimized Score: 1.0
Forced Choiced Greedy Algorithm Score: 1.0
3 Line Data:


  0%|          | 0/12 [00:00<?, ?it/s]

...

Top 1 Score: 0.0
Top 3 Score: 0.16666666666666666
Forced Choiced Combinatorially Optimized Score: 0.0
Forced Choiced Greedy Algorithm Score: 0.0

Slovak
1 Line Data:


  0%|          | 0/12 [00:00<?, ?it/s]

Top 1 Score: 0.08333333333333333
Top 3 Score: 0.25
Forced Choiced Combinatorially Optimized Score: 0.16666666666666666
Forced Choiced Greedy Algorithm Score: 0.08333333333333333
3 Line Data:


  0%|          | 0/12 [00:00<?, ?it/s]

Top 1 Score: 0.08333333333333333
Top 3 Score: 0.16666666666666666
Forced Choiced Combinatorially Optimized Score: 0.0
Forced Choiced Greedy Algorithm Score: 0.0



In [ ]:
tokenizer = AutoTokenizer.from_pretrained("lchaloupsky/czech-gpt2-oscar")
model = AutoModelForCausalLM.from_pretrained("lchaloupsky/czech-gpt2-oscar").cuda()
run_tests(model, tokenizer)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("spital/gpt2-small-czech-cs")
model = AutoModelForCausalLM.from_pretrained("spital/gpt2-small-czech-cs").cuda()
run_tests(model, tokenizer)